# [7.1] Logit Lens, Tuned Lens, and Patchscopes - Exercises

Implement local activation-to-language primitives. The CUDA report uses pinned TransformerLens `gelu-1l`, but these exercises are CPU-safe and focus on exact lens, activation-insertion Patchscope, counterfactual, and random-control contracts.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part1_lenses_patchscopes"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_lenses_patchscopes.tests as tests

PatchscopeTemplate = Literal["entity", "next_token", "fact"]


@dataclass(frozen=True)
class LensAccuracyReport:
    logit_lens_accuracy: float
    tuned_lens_accuracy: float
    improvement: float
    tuned_lens_improves: bool


@dataclass(frozen=True)
class PatchscopeAccuracyReport:
    patchscope_accuracy: float
    text_only_accuracy: float
    improvement: float
    beats_text_only: bool


@dataclass(frozen=True)
class CounterfactualActivationReport:
    original_answer: int
    patched_answer: int
    changed: bool


@dataclass(frozen=True)
class RandomActivationConfidenceReport:
    mean_confidence: float
    max_confidence: float
    passes_low_confidence: bool


## Logit Lens

Project residual activations directly through the unembedding and report top token ids plus probabilities.


In [ ]:
def logit_lens(residual_stream: t.Tensor, unembedding: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def top_tokens(logits: t.Tensor, *, k: int = 5) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


tests.test_logit_lens_and_top_tokens_match_reference(logit_lens, top_tokens)


## Tuned Lens

Apply an affine correction before unembedding and compare held-out target accuracy against ordinary logit lens.


In [ ]:
def tuned_lens(
    residual_stream: t.Tensor,
    lens_weight: t.Tensor,
    lens_bias: t.Tensor | None,
    unembedding: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


def prediction_accuracy(logits: t.Tensor, target_token_ids: t.Tensor) -> float:
    raise NotImplementedError()


def lens_accuracy_report(
    logit_lens_logits: t.Tensor,
    tuned_lens_logits: t.Tensor,
    target_token_ids: t.Tensor,
) -> LensAccuracyReport:
    raise NotImplementedError()


tests.test_tuned_lens_improves_over_logit_lens_on_toy_targets(
    logit_lens,
    tuned_lens,
    lens_accuracy_report,
)


## Attention Lens

Decode what an attention query reads: apply the attention pattern to value vectors, then project through the unembedding.


In [ ]:
def attention_lens(
    attention_pattern: t.Tensor,
    value_vectors: t.Tensor,
    unembedding: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_attention_lens_decodes_attention_weighted_values(attention_lens)


## Patchscopes

Render explicit templates and compare activation-conditioned answers against a text-only baseline on the same target ids.


In [ ]:
def patchscope_prompt(template: PatchscopeTemplate, placeholder: str = "<ACT>") -> str:
    raise NotImplementedError()


def patchscope_accuracy_report(
    patchscope_logits: t.Tensor,
    text_only_logits: t.Tensor,
    target_answer_ids: t.Tensor,
) -> PatchscopeAccuracyReport:
    raise NotImplementedError()


tests.test_patchscope_templates_and_accuracy_report(
    patchscope_prompt,
    patchscope_accuracy_report,
)


## Controls

Counterfactual activations should change decoded answers, while random activations should stay low-confidence.


In [ ]:
def counterfactual_activation_report(
    original_logits: t.Tensor,
    patched_logits: t.Tensor,
) -> CounterfactualActivationReport:
    raise NotImplementedError()


def random_activation_confidence_report(
    random_logits: t.Tensor,
    *,
    max_allowed_confidence: float = 0.6,
) -> RandomActivationConfidenceReport:
    raise NotImplementedError()


tests.test_counterfactual_and_random_activation_controls(
    counterfactual_activation_report,
    random_activation_confidence_report,
)


## Full Verification

After filling in the notebook, compare your implementation to `solutions.py`. The full CUDA path is run separately by `solutions.run_gpu_test(max_vram_gb=24.0)` and should report held-out tuned-lens improvement, Patchscope beating text-only, counterfactual answer change, random low-confidence control, attention-lens finiteness, and peak VRAM.


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
